# 03 · Tracing — the step-by-step path one run actually took

**There is no tracing anywhere in this lab today.** Notebook 01 counts what a run
spent; notebook 02 keeps what it produced. Neither records the *path* — which
steps ran, in what order, what each one saw, which branch it took, and why. When
a run comes out wrong, that is the only thing worth having. This notebook builds
it from scratch.

Deliberately plain Python, about sixty lines, no dependencies. **No
OpenTelemetry, no LangSmith, no Langfuse, no vendor SDK** — not because they are
bad, but because a reader cannot learn what a trace *is* from an integration.
A structured log of every decision an orchestrator made — step name, inputs,
chosen branch, why, duration, cost — is the whole idea, and it fits in one cell.

This is the **tracing** level: the most expensive of the three to write down and
the only one that answers *why did it do that?*

**No API key is needed.** The multi-step run below is a toy decision sequence
with stubbed steps. A trace is a record of control flow; it does not need a real
model behind it to be a real trace.

## What this notebook demonstrates

| Name | What it does | One example |
|---|---|---|
| `Span` | One step: name, inputs, branch, why, duration, cost, parent | `Span(name="route", branch="narrow")` |
| `Tracer.step` | Context manager that times a step and nests it under the one above | `with tracer.step("retrieve", query=q) as s:` |
| `s.decide(branch, why)` | Records which way a step went **and the reason**, at the moment it went | `s.decide("abstain", why="0 papers survived the filter")` |
| `s.spend(model, usd)` | Attaches a cost to the step that incurred it | `s.spend("gpt-4o-mini", 0.0004)` |
| `Tracer.tree` | The whole run printed as an indented tree | Step 6's output |
| `Tracer.find` / `Tracer.why` | Query the trace: which step chose this, and on what grounds | `tracer.why("abstain")` |
| `Tracer.as_dict` | The trace as JSON, saved beside the run's other artifacts | `runs/demo-observe-trace/trace.json` |

## Step 1 — locate the repo root and import `nbio`

Jupyter starts a kernel with its working directory set to the notebook's own
folder, two levels below the repo root, so a bare `import nbio` fails. Walk up
until `nbio.py` is found, then import it.

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import nbio
nbio.bootstrap()

## Step 2 — what a span has to carry

One span per step. Six fields earn their place:

- `name` — what step this is
- `inputs` — what it was given, so the step can be re-run by hand
- `branch` — which way it went
- `why` — the reason it went that way, recorded *at the decision*, not
  reconstructed afterwards
- `duration_ms` — where the wall clock went
- `cost_usd` / `model` — where the money went

Plus `depth` and `parent`, which are what make it a tree rather than a list.
`why` is the field that distinguishes a trace from a log. A log line says
`branch=narrow`. A span says `branch=narrow because 2 of 3 required keywords
matched`, and that sentence is what you read six weeks later.

In [ ]:
from dataclasses import dataclass, field, asdict


@dataclass
class Span:
    name: str
    depth: int = 0
    parent: str | None = None
    inputs: dict = field(default_factory=dict)
    branch: str | None = None
    why: str | None = None
    outputs: dict = field(default_factory=dict)
    duration_ms: float = 0.0
    cost_usd: float = 0.0
    model: str | None = None
    error: str | None = None

    def decide(self, branch: str, why: str) -> None:
        """Record which way this step went, and on what grounds."""
        self.branch, self.why = branch, why

    def spend(self, model: str, cost_usd: float) -> None:
        self.model, self.cost_usd = model, self.cost_usd + cost_usd

    def result(self, **outputs) -> None:
        self.outputs.update(outputs)


s = Span(name="demo")
s.decide("narrow", why="2 of 3 required keywords matched")
s.spend("gpt-4o-mini", 0.0004)
print(asdict(s))

assert s.branch == "narrow" and s.why.startswith("2 of 3")
assert s.cost_usd == 0.0004

## Step 3 — the recorder

`Tracer.step()` is a context manager: it opens a span, times it, nests it under
whatever step is currently open, and closes it on the way out — including when
the step raises, which is exactly the case where you most want the span kept.
The stack of open steps is what produces the tree.

In [ ]:
import time
import contextlib


class Tracer:
    """A structured log of every step and every decision in one run."""

    def __init__(self, run_id: str):
        self.run_id = run_id
        self.spans: list[Span] = []
        self._stack: list[Span] = []

    @contextlib.contextmanager
    def step(self, name: str, **inputs):
        span = Span(
            name=name,
            depth=len(self._stack),
            parent=self._stack[-1].name if self._stack else None,
            inputs=inputs,
        )
        # Appended on entry, not on exit, so the order is the order things
        # started -- a trace that reorders itself on completion is unreadable.
        self.spans.append(span)
        self._stack.append(span)
        t0 = time.perf_counter()
        try:
            yield span
        except Exception as exc:
            span.error = f"{type(exc).__name__}: {exc}"
            raise
        finally:
            span.duration_ms = round((time.perf_counter() - t0) * 1000, 3)
            self._stack.pop()

    @property
    def total_cost_usd(self) -> float:
        return sum(s.cost_usd for s in self.spans)

    def find(self, name: str) -> list[Span]:
        return [s for s in self.spans if s.name == name]

    def why(self, branch: str) -> list[tuple]:
        """Every step that took this branch, with the reason it gave."""
        return [(s.name, s.why) for s in self.spans if s.branch == branch]

    def as_dict(self) -> dict:
        return {
            "run_id": self.run_id,
            "n_spans": len(self.spans),
            "total_cost_usd": round(self.total_cost_usd, 6),
            "total_ms": round(sum(s.duration_ms for s in self.spans if s.depth == 0), 3),
            "spans": [asdict(s) for s in self.spans],
        }


print("Tracer:", [m for m in vars(Tracer) if not m.startswith("_")])

## Step 4 — printing it as a tree

Indent by depth, one line per span, cost and duration on the right, and the
reason on its own continuation line where there is one. The reason is the point
of the whole exercise, so it gets the room.

In [ ]:
def tree(tracer: Tracer, show_inputs: bool = False) -> None:
    print(f"trace {tracer.run_id}  —  {len(tracer.spans)} span(s), "
          f"${tracer.total_cost_usd:.6f}")
    print("-" * 78)
    for s in tracer.spans:
        pad = "  " * s.depth
        head = f"{pad}{'└─ ' if s.depth else ''}{s.name}"
        meta = f"{s.duration_ms:8.2f} ms  ${s.cost_usd:.6f}"
        print(f"{head:<50}{meta:>28}")
        if show_inputs and s.inputs:
            print(f"{pad}     in   : {s.inputs}")
        if s.branch:
            print(f"{pad}     chose: {s.branch}")
            print(f"{pad}     why  : {s.why}")
        if s.outputs:
            print(f"{pad}     out  : {s.outputs}")
        if s.error:
            print(f"{pad}     ERROR: {s.error}")


Tracer.tree = tree   # attached so the toy run below reads as tracer.tree()
print("tree() attached to Tracer")

## Step 5 — a toy orchestrator with real branches

Five steps, each of which could have gone another way: classify the question,
pick a source, filter what came back, decide whether there is enough to answer,
then answer or abstain. The steps are stubs — no model, no network — but the
*decisions* are real branches on real inputs, which is all a trace records.

Note that every `decide()` states its grounds in terms of the numbers it just
saw. A `why` that says "not enough evidence" is worthless; one that says "0 of 4
papers survived the year filter (cutoff 2021)" is the answer to a question you
have not asked yet.

In [ ]:
CORPUS = [
    {"id": "P1", "title": "Enzymatic debridement in partial-thickness burns", "year": 2015, "score": 0.82},
    {"id": "P2", "title": "Early excision timing and graft take", "year": 2018, "score": 0.77},
    {"id": "P3", "title": "Dressing choice in paediatric scalds", "year": 2016, "score": 0.61},
    {"id": "P4", "title": "Cost analysis of burn debridement pathways", "year": 2019, "score": 0.44},
]

# The configured freshness filter. This is the value the run goes wrong on.
YEAR_CUTOFF = 2021
MIN_PAPERS_TO_ANSWER = 2


def run_agent(question: str, tracer: Tracer) -> dict:
    with tracer.step("agent", question=question) as root:
        with tracer.step("classify", question=question) as sp:
            clinical = any(w in question.lower() for w in ("burn", "graft", "debridement"))
            sp.decide("clinical" if clinical else "general",
                      why=f"clinical keyword present: {clinical}")

        with tracer.step("select_source", classified=sp.branch) as sp_src:
            source = "literature_index" if sp.branch == "clinical" else "web_search"
            sp_src.decide(source, why=f"{sp.branch} questions route to {source}")
            sp_src.spend("openai/gpt-oss-120b", 0.000180)

        with tracer.step("retrieve", source=source, query=question) as sp_ret:
            candidates = sorted(CORPUS, key=lambda p: -p["score"])
            sp_ret.decide("hit", why=f"{len(candidates)} candidate(s) returned by {source}")
            sp_ret.result(candidate_ids=[p["id"] for p in candidates])

        with tracer.step("filter", cutoff=YEAR_CUTOFF, n_in=len(candidates)) as sp_f:
            kept = [p for p in candidates if p["year"] >= YEAR_CUTOFF]
            dropped = [f"{p['id']}({p['year']})" for p in candidates if p not in kept]
            sp_f.decide(f"kept {len(kept)}/{len(candidates)}",
                        why=f"year >= {YEAR_CUTOFF}; dropped {dropped}")
            sp_f.result(kept_ids=[p["id"] for p in kept])

        with tracer.step("decide_answer", n_kept=len(kept), threshold=MIN_PAPERS_TO_ANSWER) as sp_d:
            if len(kept) >= MIN_PAPERS_TO_ANSWER:
                sp_d.decide("answer", why=f"{len(kept)} papers >= threshold {MIN_PAPERS_TO_ANSWER}")
                sp_d.spend("gpt-4o-mini", 0.000920)
                out = {"status": "answered", "citations": [p["id"] for p in kept]}
            else:
                sp_d.decide("abstain",
                            why=f"only {len(kept)} paper(s) left, threshold is {MIN_PAPERS_TO_ANSWER}")
                out = {"status": "abstained", "citations": []}

        root.result(**out)
        return out


print("orchestrator defined —", MIN_PAPERS_TO_ANSWER, "papers needed, cutoff year", YEAR_CUTOFF)

## Step 6 — run it, and look at the trace

The run below abstains. On the face of it that looks like a retrieval failure:
the question is squarely in the corpus, and the corpus has four relevant papers.

In [ ]:
tracer = Tracer(run_id="demo-observe-trace")
result = run_agent("Does enzymatic debridement improve graft take in burns?", tracer)

print("OUTCOME:", result)
print()
tracer.tree()

assert result["status"] == "abstained"
assert len(tracer.spans) == 6          # agent + five steps
assert tracer.spans[0].name == "agent" and tracer.spans[0].depth == 0
assert all(s.duration_ms >= 0 for s in tracer.spans)

## Step 7 — the payoff: "why did it do that?"

The outcome looks wrong. Without a trace, the debugging starts with guesses —
is the index empty, is the query mis-embedded, is the threshold too high, did
retrieval time out? Each of those is a code-reading session.

With the trace, it is two lookups. `retrieve` says four candidates came back, so
retrieval is fine. `filter` says all four were dropped, and names the rule that
dropped them and the years it dropped them on. The abstention is correct
behaviour executing on a misconfigured freshness cutoff — a config bug, three
lines below, not a retrieval bug.

In [ ]:
print("why did it abstain?")
for name, why in tracer.why("abstain"):
    print(f"  {name}: {why}")

retrieved = tracer.find("retrieve")[0]
filtered = tracer.find("filter")[0]

print()
print(f"  retrieve returned : {len(retrieved.outputs['candidate_ids'])} "
      f"-> {retrieved.outputs['candidate_ids']}")
print(f"  filter kept       : {len(filtered.outputs['kept_ids'])} -> {filtered.outputs['kept_ids']}")
print(f"  filter's reason   : {filtered.why}")
print()
print("  verdict: retrieval worked. The year cutoff removed everything, and the")
print(f"           abstention was correct behaviour on a bad config ({YEAR_CUTOFF}).")

# The trace, not the outcome, is what carries this.
assert retrieved.outputs["candidate_ids"] == ["P1", "P2", "P3", "P4"]   # retrieval was fine
assert filtered.outputs["kept_ids"] == []                              # the filter emptied it
assert str(YEAR_CUTOFF) in filtered.why                                # the rule names itself
assert tracer.why("abstain")[0][0] == "decide_answer"

## Step 8 — fix the config, re-run, and diff the two traces

Same question, same corpus, same code — one changed cutoff. Running both and
comparing span by span is what turns a trace from a debugging aid into a
regression check: not "the answer changed", but *which step* changed and what
reason it gave this time.

In [ ]:
YEAR_CUTOFF = 2015          # the fix: the corpus predates the old cutoff entirely

fixed = Tracer(run_id="demo-observe-trace-fixed")
result_fixed = run_agent("Does enzymatic debridement improve graft take in burns?", fixed)

print("OUTCOME:", result_fixed)
print()
fixed.tree()

print()
print("span-by-span diff of the two runs:")
rows = []
for before, after in zip(tracer.spans, fixed.spans):
    changed = "CHANGED" if before.branch != after.branch else ""
    rows.append((before.name, before.branch or "-", after.branch or "-", changed))
nbio.table(rows, ("step", "before", "after", ""))

assert result_fixed["status"] == "answered"
assert len(result_fixed["citations"]) == 4
# Only the two steps downstream of the cutoff changed; routing and retrieval did not.
changed_steps = [b.name for b, a in zip(tracer.spans, fixed.spans) if b.branch != a.branch]
assert changed_steps == ["filter", "decide_answer"]

## Step 9 — cost, per step

The counting level from notebook 01 tells you a run cost a tenth of a cent. The
trace tells you *which step* spent it. On a run that is too expensive, that is
the difference between "use a cheaper model" and "the source-selection step is
making an LLM call it does not need".

In [ ]:
spend_rows = [
    (s.name, s.model or "-", f"${s.cost_usd:.6f}", f"{s.duration_ms:.2f} ms")
    for s in fixed.spans if s.cost_usd or s.model
]
nbio.table(spend_rows, ("step", "model", "cost", "duration"))
print()
print(f"run total: ${fixed.total_cost_usd:.6f} across {len(spend_rows)} paid step(s)")

# The same totals nbio's meter would have produced, replayed from the trace.
with nbio.cost_meter(budget_usd=0.05) as meter:
    for s in fixed.spans:
        if s.model:
            meter.record_cost(s.model, s.cost_usd)
print()
print(meter.report())

assert abs(meter.cost_usd - fixed.total_cost_usd) < 1e-12
assert meter.calls == len(spend_rows)

## Step 10 — save the trace beside the run's other artifacts

A trace that lives only in a kernel dies with the kernel. Written to
`runs/<run_id>/trace.json`, it sits next to the manifest, papers and answer from
`02-run-artifacts.ipynb` — same directory, same discipline. `nbio.load_run()`
only knows about the three standard files, so the trace is read back with a
plain `json.loads`; the path convention is what ties them together.

In [ ]:
import json as jsonlib

repo_root = nbio.bootstrap()
trace_dir = nbio.runs_dir() / fixed.run_id
trace_dir.mkdir(parents=True, exist_ok=True)
trace_path = trace_dir / "trace.json"
trace_path.write_text(jsonlib.dumps(fixed.as_dict(), ensure_ascii=False, indent=2), encoding="utf-8")

print("written:", trace_path.relative_to(repo_root))

reloaded = jsonlib.loads(trace_path.read_text(encoding="utf-8"))
print(f"reloaded {reloaded['n_spans']} span(s), ${reloaded['total_cost_usd']:.6f}, "
      f"{reloaded['total_ms']:.2f} ms")
print()
nbio.show_json(reloaded["spans"][4], limit=700)

assert reloaded["n_spans"] == len(fixed.spans)
assert reloaded["spans"][4]["why"] == fixed.spans[4].why    # the reason survives the round trip
assert abs(reloaded["total_cost_usd"] - fixed.total_cost_usd) < 1e-6

## Where this runs in the pipeline

Around the orchestrator, wherever one lives. `01-modules/04-orchestrate` is the
stage that will have one; until it does, the recorder above is stage-agnostic on
purpose — `tracer.step(...)` wraps any function, and the trace is only as good
as the `why` strings the code passes in.

The three levels, in one line each: **counting** says what it cost,
**artifacts** say what it produced, **tracing** says why it did that. The first
two are cheap and you should always have them. The third is the one you wish you
had turned on before the run that went wrong.

## What did not come across

- **Distributed tracing.** One process, one thread, one in-memory list. No trace
  context propagated across a network boundary, no sampling, no collector. A
  trace that has to survive a hop between services is what OpenTelemetry is for,
  and this is not a substitute for it.
- **Automatic instrumentation.** Every span here is opened by hand. Frameworks
  wrap your calls and produce spans for free; the cost is that they produce the
  spans *they* chose, and none of them can invent the `why` string — that only
  exists because the code writing the branch also wrote the reason.
- **Async and concurrency.** `Tracer._stack` is a plain list, so two coroutines
  tracing at once would interleave into one nonsensical tree. A `contextvars`
  stack is the fix; it is left out because nothing here runs concurrently.
- **Prompt and response bodies.** Spans record decisions, not payloads. Storing
  the full prompt and completion per step is what makes a trace replayable — and
  is also how partner data ends up in a git repo, which is why it is not done
  here.
- **A real orchestrator.** The toy above has five steps and no loop. A real agent
  revisits steps, retries, and backtracks, which makes trace depth and span
  count the first things you have to bound.
- **Timing under load.** `duration_ms` is wall clock on stub functions, so the
  numbers above are microseconds of Python, not of work. They prove the plumbing
  measures something; they say nothing about what a real step costs.